In [ ]:
# In[1]:

import pandas as pd

# candidate services
candidates = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]

# load csvs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_err = pd.read_csv("error_logs.csv")

# parse timestamps to UTC datetime (Unix seconds)
for df in (df_metric, df_trace, df_log, df_err):
    df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# helper to build compact summary per dataframe
def build_summary(df, filename, name_col=None):
    rows = len(df)
    earliest = df['ts'].min()
    latest = df['ts'].max()
    unique_cmdb = sorted(df['cmdb_id'].dropna().unique().tolist())
    unique_cmdb_sample = unique_cmdb[:20]
    intersection = sorted(list(set(unique_cmdb) & set(candidates)))
    # unique names for metric/trace/log
    if name_col and name_col in df.columns:
        unique_names = sorted(df[name_col].dropna().unique().tolist())
        unique_names_sample = unique_names[:20]
    else:
        unique_names_sample = []
    return {
        "file": filename,
        "rows": rows,
        "earliest_ts_utc": earliest,
        "latest_ts_utc": latest,
        "unique_cmdb_count": len(unique_cmdb),
        "unique_cmdb_sample": unique_cmdb_sample,
        "intersection_with_candidates": intersection,
        "unique_names_sample": unique_names_sample
    }

s_metric = build_summary(df_metric, "metric.csv", name_col="kpi_name")
s_trace  = build_summary(df_trace,  "trace.csv",  name_col="trace_name")
s_log    = build_summary(df_log,    "log.csv",    name_col="log_name")
s_err    = build_summary(df_err,    "error_logs.csv", name_col=None)

summary_df = pd.DataFrame([s_metric, s_trace, s_log, s_err])

# Keep output compact: selected columns only, up to 20 items already enforced
summary_df[["file","rows","earliest_ts_utc","latest_ts_utc","unique_cmdb_count","unique_cmdb_sample","intersection_with_candidates","unique_names_sample"]]

```
Out[1]:
```
```python
# Compose a concise plain-English summary using the previously computed summaries (s_metric, s_trace, s_log, s_err).
def ts_to_str(t):
    return "N/A" if pd.isna(t) else t.isoformat()

def sample_list(lst, n=6):
    return lst[:n] if lst else []

summary = (
    "Summary of telemetry files (timestamps in UTC):\n\n"
    f"metric.csv: {s_metric['rows']} rows; time range {ts_to_str(s_metric['earliest_ts_utc'])} -> {ts_to_str(s_metric['latest_ts_utc'])}; "
    f"{s_metric['unique_cmdb_count']} unique cmdb_id (sample: {sample_list(s_metric['unique_cmdb_sample'])}); "
    f"candidate services present: {s_metric['intersection_with_candidates']}; "
    f"example kpi_names: {sample_list(s_metric['unique_names_sample'])}.\n\n"
    f"trace.csv: {s_trace['rows']} rows; time range {ts_to_str(s_trace['earliest_ts_utc'])} -> {ts_to_str(s_trace['latest_ts_utc'])}; "
    f"{s_trace['unique_cmdb_count']} unique cmdb_id (sample: {sample_list(s_trace['unique_cmdb_sample'])}); "
    f"candidate services present: {s_trace['intersection_with_candidates']}; "
    f"example trace_names: {sample_list(s_trace['unique_names_sample'])}.\n\n"
    f"log.csv: {s_log['rows']} rows; time range {ts_to_str(s_log['earliest_ts_utc'])} -> {ts_to_str(s_log['latest_ts_utc'])}; "
    f"{s_log['unique_cmdb_count']} unique cmdb_id (sample: {sample_list(s_log['unique_cmdb_sample'])}); "
    f"candidate services present: {s_log['intersection_with_candidates']}; "
    f"example log_names: {sample_list(s_log['unique_names_sample'])}.\n\n"
    f"error_logs.csv: {s_err['rows']} rows; time range {ts_to_str(s_err['earliest_ts_utc'])} -> {ts_to_str(s_err['latest_ts_utc'])}; "
    "no cmdb_ids present.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

file   rows           earliest_ts_utc             latest_ts_utc  unique_cmdb_count                                 unique_cmdb_sample                       intersection_with_candidates                                unique_names_sample
0      metric.csv   9155 2024-01-23 18:50:00+00:00 2024-01-23 19:14:00+00:00                 68  [ts-admin-basic-info-service, ts-admin-order-s...  [ts-auth-service, ts-order-service, ts-route-s...  [cpu, diskio, error, latency-50, latency-90, m...
1       trace.csv  18696 2024-01-23 18:50:00+00:00 2024-01-23 19:14:00+00:00                 28  [root, ts-admin-basic-info-service, ts-admin-t...  [ts-auth-service, ts-order-service, ts-route-s...  [trace.from_root.duration_mean, trace.from_roo...
2         log.csv   2196 2024-01-23 18:50:00+00:00 2024-01-23 19:14:00+00:00                 46  [ts-admin-basic-info-service, ts-admin-travel-...  [ts-auth-service, ts-order-service, ts-route-s...                   [log.error_count, log.row_count]
3  error_logs.csv      0                       NaT                       NaT                  0                                                 []                                                 []                                                 []```
```

In [ ]:
# In[2]:

# reuse existing variables: df_metric, df_trace, df_log, candidates, and parsed 'ts' column
import pandas as pd

# Incident window (UTC)
start = pd.to_datetime("2024-01-23 18:47:25", utc=True)
end   = pd.to_datetime("2024-01-23 19:17:25", utc=True)

def process(df, source, name_col):
    # restrict to candidate services
    df_cand = df[df['cmdb_id'].isin(candidates)].copy()
    # present candidate cmdb_ids for this file
    present = sorted(df_cand['cmdb_id'].dropna().unique().tolist())
    if len(df_cand)==0:
        return pd.DataFrame(columns=["source","cmdb_id","name","global_p95","total_points",
                                     "points_in_window","mean_value_in_window","max_value_in_window","earliest_anom_ts_utc"]), present

    # compute global p95 and total_points on the full available series (rule 9)
    grp = df_cand.groupby(['cmdb_id', name_col])['value']
    agg_full = grp.agg(global_p95=lambda x: x.quantile(0.95), total_points='count').reset_index()

    # windowed slice
    df_win = df_cand[(df_cand['ts'] >= start) & (df_cand['ts'] <= end)].copy()

    # compute window aggregates
    if len(df_win)>0:
        win_agg = df_win.groupby(['cmdb_id', name_col])['value'].agg(
            points_in_window='count', mean_value_in_window='mean', max_value_in_window='max'
        ).reset_index()
    else:
        win_agg = pd.DataFrame(columns=['cmdb_id', name_col, 'points_in_window','mean_value_in_window','max_value_in_window'])

    # earliest anomaly: need to compare window values to global_p95
    if len(df_win)>0:
        # merge global_p95 into window rows
        df_win = df_win.merge(agg_full[['cmdb_id', name_col, 'global_p95']], on=['cmdb_id', name_col], how='left')
        anom_rows = df_win[df_win['value'] > df_win['global_p95']].copy()
        if len(anom_rows)>0:
            anom_agg = anom_rows.groupby(['cmdb_id', name_col])['ts'].min().reset_index().rename(columns={'ts':'earliest_anom_ts_utc'})
        else:
            anom_agg = pd.DataFrame(columns=['cmdb_id', name_col, 'earliest_anom_ts_utc'])
    else:
        anom_agg = pd.DataFrame(columns=['cmdb_id', name_col, 'earliest_anom_ts_utc'])

    # merge full agg with window agg and anomaly times
    merged = agg_full.merge(win_agg, on=['cmdb_id', name_col], how='left').merge(anom_agg, on=['cmdb_id', name_col], how='left')

    # normalize columns and fill missing window counts with 0
    merged['points_in_window'] = merged['points_in_window'].fillna(0).astype(int)
    merged['mean_value_in_window'] = merged['mean_value_in_window']  # keep NaN if no points
    merged['max_value_in_window'] = merged['max_value_in_window']
    # convert earliest_anom to ISO string or None
    merged['earliest_anom_ts_utc'] = merged['earliest_anom_ts_utc'].apply(lambda x: None if pd.isna(x) else x.isoformat())

    # rename name_col to 'name' for unified output
    merged = merged.rename(columns={name_col:'name'})
    merged['source'] = source

    # select and order columns
    merged = merged[['source','cmdb_id','name','global_p95','total_points',
                     'points_in_window','mean_value_in_window','max_value_in_window','earliest_anom_ts_utc']]
    return merged, present

# process each file
res_metric, present_metric = process(df_metric, 'metric', 'kpi_name')
res_trace,  present_trace  = process(df_trace,  'trace',  'trace_name')
res_log,    present_log    = process(df_log,    'log',    'log_name')

# combine
combined = pd.concat([res_metric, res_trace, res_log], ignore_index=True)

# sort by points_in_window desc then max_value_in_window desc, limit top 20
combined_sorted = combined.sort_values(by=['points_in_window','max_value_in_window'], ascending=[False, False]).reset_index(drop=True)
top20 = combined_sorted.head(20)

# present candidate lists per file as a small DataFrame
present_df = pd.DataFrame({
    'file': ['metric.csv','trace.csv','log.csv'],
    'present_candidates': [present_metric, present_trace, present_log]
})

# Display compact outputs: top20 aggregated table and present candidates per file
top20, present_df

```
Out[2]:
```
```python
# Build a concise plain-English summary from the previously computed results (top20, present_df).
lines = []
lines.append("Telemetry summary (incident window 2024-01-23 18:47:25 -> 2024-01-23 19:17:25 UTC):")
lines.append("")

# present candidate services per file
for _, r in present_df.iterrows():
    lines.append(f"- {r['file']}: candidate services present: {r['present_candidates']}")

lines.append("")
lines.append("Top findings (top rows by points_in_window then max_value_in_window):")
# show up to top 10 rows with key fields
for i, row in top20.head(10).iterrows():
    src = row['source']
    cmdb = row['cmdb_id']
    name = row['name']
    pts = int(row['points_in_window'])
    maxv = row['max_value_in_window']
    # concise numeric formatting
    maxv_s = "None" if pd.isna(maxv) else f"{maxv:.6g}"
    p95_s = f"{row['global_p95']:.6g}"
    earliest = row['earliest_anom_ts_utc'] if row['earliest_anom_ts_utc'] is not None else "None"
    lines.append(f"  {i+1}. [{src}] {cmdb} | {name} — points_in_window={pts}, max={maxv_s}, global_p95={p95_s}, earliest_anom={earliest}")

lines.append("")
lines.append("Interpretation:")
lines.append("- All five candidate services (ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service) are present in metric, trace, and log telemetry.")
lines.append("- Memory ('mem') is the top metric for all five services within the window (each has 25 points in-window) and shows values exceeding the global 95th percentile; earliest anomalous timestamps per service are listed above.")
lines.append("- Disk I/O ('diskio') is elevated for several services (notably ts-auth-service, ts-order-service, ts-train-service, ts-travel-service) with anomalies appearing early in the window.")
lines.append("- Trace and log row_count metrics also show large counts and exceed their thresholds for multiple services (see trace/log entries above).")
lines.append("")
lines.append("Concise conclusion: multiple candidate services exhibit memory and disk I/O elevations within the incident window; memory anomalies across all five services are the most prominent signal.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    source            cmdb_id                                     name    global_p95  total_points  points_in_window  mean_value_in_window  max_value_in_window       earliest_anom_ts_utc
0   metric  ts-travel-service                                      mem  2.763990e+08            25                25          2.760885e+08         2.766280e+08  2024-01-23T18:53:00+00:00
1   metric   ts-order-service                                      mem  2.647648e+08            25                25          2.639241e+08         2.648420e+08  2024-01-23T19:13:00+00:00
2   metric   ts-route-service                                      mem  2.590723e+08            25                25          2.577788e+08         2.595550e+08  2024-01-23T19:01:00+00:00
3   metric    ts-auth-service                                      mem  2.576193e+08            25                25          2.529535e+08         2.577902e+08  2024-01-23T19:12:00+00:00
4   metric   ts-train-service                                      mem  2.528442e+08            25                25          2.525048e+08         2.529311e+08  2024-01-23T19:13:00+00:00
5   metric    ts-auth-service                                   diskio  8.425506e+04            25                25          5.921901e+04         1.908668e+05  2024-01-23T18:50:00+00:00
6   metric  ts-travel-service                                   diskio  6.674097e+04            25                25          5.355164e+04         7.268996e+04  2024-01-23T18:51:00+00:00
7   metric   ts-order-service                                   diskio  6.038903e+04            25                25          5.078770e+04         7.100172e+04  2024-01-23T18:50:00+00:00
8   metric   ts-train-service                                   diskio  5.983550e+04            25                25          5.098192e+04         6.118910e+04  2024-01-23T18:54:00+00:00
9   metric   ts-route-service                                   diskio  5.942860e+04            25                25          5.058177e+04         6.061393e+04  2024-01-23T18:58:00+00:00
10   trace   ts-train-service    trace.from_ts-train-service.row_count  2.708800e+03            25                25          2.287840e+03         2.760000e+03  2024-01-23T18:54:00+00:00
11   trace   ts-train-service      trace.to_ts-train-service.row_count  2.708800e+03            25                25          2.287840e+03         2.760000e+03  2024-01-23T18:54:00+00:00
12   trace   ts-route-service    trace.from_ts-route-service.row_count  2.127200e+03            25                25          1.858160e+03         2.180000e+03  2024-01-23T18:54:00+00:00
13   trace   ts-route-service      trace.to_ts-route-service.row_count  2.127200e+03            25                25          1.858160e+03         2.180000e+03  2024-01-23T18:54:00+00:00
14     log  ts-travel-service                            log.row_count  2.071600e+03            25                25          1.557240e+03         2.124000e+03  2024-01-23T18:55:00+00:00
15   trace  ts-travel-service   trace.from_ts-travel-service.row_count  1.851000e+03            25                25          1.383040e+03         1.904000e+03  2024-01-23T18:55:00+00:00
16   trace  ts-travel-service     trace.to_ts-travel-service.row_count  1.851000e+03            25                25          1.383040e+03         1.904000e+03  2024-01-23T18:55:00+00:00
17     log   ts-route-service                            log.row_count  1.085400e+03            25                25          9.360000e+02         1.151000e+03  2024-01-23T18:55:00+00:00
18   trace   ts-route-service  trace.from_ts-travel2-service.row_count  6.048000e+02            25                25          5.356400e+02         6.190000e+02  2024-01-23T18:52:00+00:00
19   trace   ts-order-service    trace.from_ts-order-service.row_count  5.488000e+02            25                25          4.194000e+02         5.500000e+02  2024-01-23T18:54:00+00:00,          file                                 present_candidates
0  metric.csv  [ts-auth-service, ts-order-service, ts-route-s...
1   trace.csv  [ts-auth-service, ts-order-service, ts-route-s...
2     log.csv  [ts-auth-service, ts-order-service, ts-route-s...)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# reuse df_metric, candidates, and ts if available; otherwise load and parse timestamps
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start = pd.to_datetime("2024-01-23 18:47:25", utc=True)
end   = pd.to_datetime("2024-01-23 19:17:25", utc=True)

# Filter to candidate services and kpis mem/diskio
kpis = ['mem', 'diskio']
df_m = df_metric[df_metric['cmdb_id'].isin(candidates) & df_metric['kpi_name'].isin(kpis)].copy()

# If empty, return empty results
if df_m.empty:
    segments_top20 = pd.DataFrame(columns=['cmdb_id','kpi_name','start_ts_utc','end_ts_utc','length_minutes','max_value','global_p95','severity_ratio'])
    distinct_services_count = 0
else:
    # Compute global p95 per (cmdb_id, kpi_name) using full series (rule 9)
    gp = df_m.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'global_p95'})
    gp['global_p95'] = gp['global_p95'].astype(float)

    # Merge global_p95 back
    df_m = df_m.merge(gp, on=['cmdb_id','kpi_name'], how='left')

    # Filter to incident window
    df_win = df_m[(df_m['ts'] >= start) & (df_m['ts'] <= end)].copy()
    df_win = df_win.sort_values(['cmdb_id','kpi_name','ts'])

    segments = []
    # iterate groups
    for (cmdb, kpi), grp in df_win.groupby(['cmdb_id','kpi_name']):
        # boolean mask where value > global_p95
        gp_val = grp['global_p95'].iat[0]
        mask = grp['value'] > gp_val
        if not mask.any():
            continue
        # keep only True rows (still preserve ts order)
        true_rows = grp[mask].copy().reset_index(drop=True)
        # compute diff seconds between consecutive true rows
        true_rows['diff_s'] = true_rows['ts'].diff().dt.total_seconds().fillna(0)
        # start a new segment when diff != 60
        true_rows['new_seg'] = (true_rows['diff_s'] != 60).cumsum()
        for seg_id, sgrp in true_rows.groupby('new_seg'):
            start_ts = sgrp['ts'].min()
            end_ts = sgrp['ts'].max()
            npoints = len(sgrp)
            length_minutes = int(round((end_ts - start_ts).total_seconds() / 60.0)) + 1 if npoints>0 else 0
            max_value = sgrp['value'].max()
            # severity_ratio safe compute
            if gp_val == 0:
                severity_ratio = None
            else:
                severity_ratio = (max_value - gp_val) / gp_val
            segments.append({
                'cmdb_id': cmdb,
                'kpi_name': kpi,
                'start_ts_utc': start_ts,
                'end_ts_utc': end_ts,
                'length_minutes': length_minutes,
                'max_value': max_value,
                'global_p95': gp_val,
                'severity_ratio': severity_ratio
            })

    # assemble DataFrame
    if len(segments) == 0:
        segments_df = pd.DataFrame(columns=['cmdb_id','kpi_name','start_ts_utc','end_ts_utc','length_minutes','max_value','global_p95','severity_ratio'])
    else:
        segments_df = pd.DataFrame(segments)
        # format timestamps as ISO strings for compactness
        segments_df['start_ts_utc'] = segments_df['start_ts_utc'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
        segments_df['end_ts_utc']   = segments_df['end_ts_utc'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')

    # number of distinct services with at least one fault segment
    distinct_services_count = int(segments_df['cmdb_id'].nunique())

    # sort by severity_ratio descending, limit top 20 (place None/NaN at bottom)
    segments_df['severity_sort'] = segments_df['severity_ratio'].apply(lambda x: -np.inf if x is None else x)
    segments_top20 = segments_df.sort_values(by='severity_ratio', ascending=False).head(20).drop(columns=['severity_sort']).reset_index(drop=True)

# Final compact outputs
segments_top20, distinct_services_count

```
Out[3]:
```
Summary of fault segments in metric.csv (candidates: ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service; kpis: mem, diskio; incident window 2024-01-23 18:47:25 → 2024-01-23 19:17:25 UTC):

- Number of distinct services with ≥1 fault segment: 5

Top fault segments by severity_ratio (highest first):
1. ts-auth-service — diskio
   - window: 2024-01-23T18:50:00 → 2024-01-23T18:51:00 (2 minutes)
   - max_value = 190,866.8 ; global_p95 = 84,255.06
   - severity_ratio = 1.265 (≈ +126.5% above p95)

2. ts-order-service — diskio
   - window: 2024-01-23T18:50:00 → 2024-01-23T18:50:00 (1 minute)
   - max_value = 71,001.72 ; global_p95 = 60,389.03
   - severity_ratio = 0.176 (≈ +17.6%)

3. ts-travel-service — diskio
   - window: 2024-01-23T18:51:00 → 2024-01-23T18:51:00 (1 minute)
   - max_value = 72,689.96 ; global_p95 = 66,740.97
   - severity_ratio = 0.089 (≈ +8.9%)

4. ts-train-service — diskio
   - window: 2024-01-23T18:54:00 → 2024-01-23T18:54:00 (1 minute)
   - max_value = 61,189.10 ; global_p95 = 59,835.50
   - severity_ratio = 0.023 (≈ +2.3%)

5. ts-route-service — diskio
   - window: 2024-01-23T19:06:00 → 2024-01-23T19:06:00 (1 minute)
   - max_value = 60,613.93 ; global_p95 = 59,428.60
   - severity_ratio = 0.020 (≈ +2.0%)

6. ts-route-service — mem
   - window: 2024-01-23T19:01:00 → 2024-01-23T19:01:00 (1 minute)
   - max_value = 259,555,000 ; global_p95 = 259,072,300
   - severity_ratio = 0.0019 (≈ +0.19%)

7. ts-travel-service — mem
   - window: 2024-01-23T18:53:00 → 2024-01-23T18:54:00 (2 minutes)
   - max_value = 276,628,000 ; global_p95 = 276,399,000
   - severity_ratio = 0.00083 (≈ +0.083%)

8. ts-auth-service — mem
   - window: 2024-01-23T19:12:00 → 2024-01-23T19:13:00 (2 minutes)
   - max_value = 257,790,200 ; global_p95 = 257,619,300
   - severity_ratio = 0.00066 (≈ +0.066%)

(Additional smaller segments exist; output limited to top results by severity.)

Interpretation:
- The clearest, largest anomaly is a short but strong disk I/O spike on ts-auth-service (≈126% above its global p95) around 18:50–18:51 UTC. Other services show smaller diskio spikes (ts-order, ts-travel, ts-train, ts-route).
- Memory (mem) deviations are present for all services but the excess over global p95 is very small (fractions of a percent).
- Overall, disk I/O on ts-auth-service is the most severe single fault segment in the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id kpi_name              start_ts_utc                end_ts_utc  length_minutes     max_value    global_p95  severity_ratio
0     ts-auth-service   diskio  2024-01-23T18:50:00+0000  2024-01-23T18:51:00+0000               2  1.908668e+05  8.425506e+04        1.265346
1    ts-order-service   diskio  2024-01-23T18:50:00+0000  2024-01-23T18:50:00+0000               1  7.100172e+04  6.038903e+04        0.175739
2   ts-travel-service   diskio  2024-01-23T18:51:00+0000  2024-01-23T18:51:00+0000               1  7.268996e+04  6.674097e+04        0.089136
3    ts-train-service   diskio  2024-01-23T18:54:00+0000  2024-01-23T18:54:00+0000               1  6.118910e+04  5.983550e+04        0.022622
4    ts-route-service   diskio  2024-01-23T19:06:00+0000  2024-01-23T19:06:00+0000               1  6.061393e+04  5.942860e+04        0.019945
5    ts-train-service   diskio  2024-01-23T19:07:00+0000  2024-01-23T19:07:00+0000               1  6.025366e+04  5.983550e+04        0.006988
6    ts-order-service   diskio  2024-01-23T18:54:00+0000  2024-01-23T18:54:00+0000               1  6.079063e+04  6.038903e+04        0.006650
7    ts-route-service   diskio  2024-01-23T18:58:00+0000  2024-01-23T18:58:00+0000               1  5.981414e+04  5.942860e+04        0.006487
8    ts-route-service      mem  2024-01-23T19:01:00+0000  2024-01-23T19:01:00+0000               1  2.595550e+08  2.590723e+08        0.001863
9   ts-travel-service   diskio  2024-01-23T19:12:00+0000  2024-01-23T19:12:00+0000               1  6.681684e+04  6.674097e+04        0.001137
10  ts-travel-service      mem  2024-01-23T18:53:00+0000  2024-01-23T18:54:00+0000               2  2.766280e+08  2.763990e+08        0.000828
11    ts-auth-service      mem  2024-01-23T19:12:00+0000  2024-01-23T19:13:00+0000               2  2.577902e+08  2.576193e+08        0.000664
12   ts-train-service      mem  2024-01-23T19:13:00+0000  2024-01-23T19:14:00+0000               2  2.529311e+08  2.528442e+08        0.000344
13   ts-order-service      mem  2024-01-23T19:13:00+0000  2024-01-23T19:14:00+0000               2  2.648420e+08  2.647648e+08        0.000291
14   ts-route-service      mem  2024-01-23T19:14:00+0000  2024-01-23T19:14:00+0000               1  2.590747e+08  2.590723e+08        0.000009, 5)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# reuse existing variables if present
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# ensure candidates and incident window exist
try:
    candidates
except NameError:
    candidates = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]

try:
    start, end
except NameError:
    start = pd.to_datetime("2024-01-23 18:47:25", utc=True)
    end   = pd.to_datetime("2024-01-23 19:17:25", utc=True)

# 1) Identify trace_name values referencing ts-auth-service
mask_auth_ref = df_trace['trace_name'].str.contains('from_ts-auth-service|to_ts-auth-service', na=False)
auth_trace_names = sorted(df_trace.loc[mask_auth_ref, 'trace_name'].dropna().unique().tolist())[:20]
auth_trace_names_df = pd.DataFrame({'auth_trace_name': auth_trace_names})

# 2) Select relevant series for candidate services:
#    include any trace_name that references ts-auth-service OR equals 'trace.self.duration_mean'
relevant_names = set(auth_trace_names)
relevant_names.add('trace.self.duration_mean')

df_relevant = df_trace[df_trace['cmdb_id'].isin(candidates) & df_trace['trace_name'].isin(relevant_names)].copy()

# If none, return empty results
if df_relevant.empty:
    aggregated = pd.DataFrame(columns=['cmdb_id','trace_name','global_p95','total_points',
                                       'points_in_window','max_value_in_window','mean_value_in_window','earliest_anom_ts_utc'])
else:
    # 2) Compute global_p95 and total_points on full series (per (cmdb_id, trace_name))
    grp_full = df_relevant.groupby(['cmdb_id','trace_name'])['value']
    agg_full = grp_full.agg(global_p95=lambda x: x.quantile(0.95), total_points='count').reset_index()
    agg_full['global_p95'] = agg_full['global_p95'].astype(float)

    # 3) Windowed aggregates
    df_win = df_relevant[(df_relevant['ts'] >= start) & (df_relevant['ts'] <= end)].copy()
    win_agg = df_win.groupby(['cmdb_id','trace_name'])['value'].agg(
        points_in_window='count',
        max_value_in_window='max',
        mean_value_in_window='mean'
    ).reset_index()

    # earliest timestamp in-window where value > global_p95
    if not df_win.empty:
        df_win = df_win.merge(agg_full[['cmdb_id','trace_name','global_p95']], on=['cmdb_id','trace_name'], how='left')
        anom = df_win[df_win['value'] > df_win['global_p95']].copy()
        if not anom.empty:
            anom_earliest = anom.groupby(['cmdb_id','trace_name'])['ts'].min().reset_index().rename(columns={'ts':'earliest_anom_ts_utc'})
        else:
            anom_earliest = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_anom_ts_utc'])
    else:
        anom_earliest = pd.DataFrame(columns=['cmdb_id','trace_name','earliest_anom_ts_utc'])

    # merge together
    aggregated = agg_full.merge(win_agg, on=['cmdb_id','trace_name'], how='left').merge(anom_earliest, on=['cmdb_id','trace_name'], how='left')

    # fill missing window counts with 0
    aggregated['points_in_window'] = aggregated['points_in_window'].fillna(0).astype(int)
    # keep mean/max as NaN if no points
    # format earliest_anom_ts_utc as ISO string or None
    aggregated['earliest_anom_ts_utc'] = aggregated['earliest_anom_ts_utc'].apply(lambda x: None if pd.isna(x) else x.isoformat())

    # reorder/select columns
    aggregated = aggregated[['cmdb_id','trace_name','global_p95','total_points',
                             'points_in_window','max_value_in_window','mean_value_in_window','earliest_anom_ts_utc']]

# 4) sort and limit to top 20 (by points_in_window desc, then max_value_in_window desc)
aggregated_sorted = aggregated.sort_values(by=['points_in_window','max_value_in_window'], ascending=[False, False]).reset_index(drop=True)
top20_trace = aggregated_sorted.head(20)

# Outputs: top20 aggregated table and list of auth-referencing trace_names
top20_trace, auth_trace_names_df

```
Out[4]:
```
```python
# Build a concise plain-English summary from the previous results (top20_trace, auth_trace_names_df)
auth_names = auth_trace_names_df['auth_trace_name'].tolist()
agg = top20_trace

# extract key facts for ts-auth-service if present
auth_agg = agg[agg['cmdb_id'] == 'ts-auth-service']
row_count_rows = auth_agg[auth_agg['trace_name'].str.contains('row_count', na=False)]
dur_p95_rows = auth_agg[auth_agg['trace_name'].str.contains('duration_p95', na=False)]
dur_mean_rows = auth_agg[auth_agg['trace_name'].str.contains('duration_mean', na=False)]
error_rate_rows = auth_agg[auth_agg['trace_name'].str.contains('error_rate', na=False)]

lines = []
lines.append("Trace analysis summary (incident window 2024-01-23 18:47:25 → 2024-01-23 19:17:25 UTC):")
lines.append("")
lines.append("Trace names referencing ts-auth-service found (up to 20):")
lines.append(f"  {auth_names}")
lines.append("")
lines.append("Aggregated results (candidate services):")
if auth_agg.empty:
    lines.append("  No relevant trace series for candidate services were present in the window.")
else:
    lines.append(f"  Observed trace series for ts-auth-service: {auth_agg['trace_name'].tolist()}")
    # row_count summary
    if not row_count_rows.empty:
        rc = row_count_rows.iloc[0]
        lines.append(f"  - Row counts (inbound/outbound): global_p95={rc['global_p95']:.2f}, max_in_window={rc['max_value_in_window']:.0f}, mean_in_window={rc['mean_value_in_window']:.1f}, earliest_anom={rc['earliest_anom_ts_utc']}")
    # duration_p95 summary
    if not dur_p95_rows.empty:
        dp = dur_p95_rows.iloc[0]
        lines.append(f"  - duration_p95: global_p95={dp['global_p95']:.5f}, max_in_window={dp['max_value_in_window']:.6f}, earliest_anom={dp['earliest_anom_ts_utc']}")
    # duration_mean summary
    if not dur_mean_rows.empty:
        dm = dur_mean_rows.iloc[0]
        lines.append(f"  - duration_mean: global_p95={dm['global_p95']:.5f}, max_in_window={dm['max_value_in_window']:.6f}, earliest_anom={dm['earliest_anom_ts_utc']}")
    # error_rate summary
    if not error_rate_rows.empty:
        er = error_rate_rows.iloc[0]
        er_anom = er['earliest_anom_ts_utc'] if er['earliest_anom_ts_utc'] is not None else "None"
        lines.append(f"  - error_rate: global_p95={er['global_p95']:.5f}, max_in_window={er['max_value_in_window']:.6f}, earliest_anom={er_anom}")

    lines.append("")
    lines.append("Interpretation:")
    lines.append("  - ts-auth-service shows elevated inbound/outbound trace row_counts (global_p95≈337.6) with max 344 and earliest anomaly at 2024-01-23T18:52:00Z.")
    lines.append("  - Trace durations (duration_p95 and duration_mean) for ts-auth-service exceed their global p95 later in the window (earliest around 19:03–19:05 UTC).")
    lines.append("  - Error rate series have p95=0 and do not show threshold-crossing anomalies in the window.")
    lines.append("")
    lines.append("Conclusion: ts-auth-service exhibits early increased traffic (row_count) around 18:52 UTC and subsequent increased trace durations ~19:03–19:05 UTC, indicating trace-level performance degradation tied to auth-related calls.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id                                trace_name  global_p95  total_points  points_in_window  max_value_in_window  mean_value_in_window       earliest_anom_ts_utc
0  ts-auth-service      trace.from_ts-auth-service.row_count   337.60000            25                25           344.000000            299.200000  2024-01-23T18:52:00+00:00
1  ts-auth-service        trace.to_ts-auth-service.row_count   337.60000            25                25           344.000000            299.200000  2024-01-23T18:52:00+00:00
2  ts-auth-service   trace.from_ts-auth-service.duration_p95     0.09891            25                25             0.100410              0.051802  2024-01-23T19:03:00+00:00
3  ts-auth-service     trace.to_ts-auth-service.duration_p95     0.09891            25                25             0.100410              0.051802  2024-01-23T19:03:00+00:00
4  ts-auth-service  trace.from_ts-auth-service.duration_mean     0.02649            25                25             0.038423              0.011579  2024-01-23T19:05:00+00:00
5  ts-auth-service    trace.to_ts-auth-service.duration_mean     0.02649            25                25             0.038423              0.011579  2024-01-23T19:05:00+00:00
6  ts-auth-service     trace.from_ts-auth-service.error_rate     0.00000            25                25             0.000000              0.000000                       None
7  ts-auth-service       trace.to_ts-auth-service.error_rate     0.00000            25                25             0.000000              0.000000                       None,                             auth_trace_name
0  trace.from_ts-auth-service.duration_mean
1   trace.from_ts-auth-service.duration_p95
2     trace.from_ts-auth-service.error_rate
3      trace.from_ts-auth-service.row_count
4    trace.to_ts-auth-service.duration_mean
5     trace.to_ts-auth-service.duration_p95
6       trace.to_ts-auth-service.error_rate
7        trace.to_ts-auth-service.row_count)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing dataframes if available, otherwise load and parse
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

try:
    df_err
except NameError:
    df_err = pd.read_csv("error_logs.csv")
    df_err['ts'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True)

# Ensure candidates and incident window exist
try:
    candidates
except NameError:
    candidates = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]

try:
    start, end
except NameError:
    start = pd.to_datetime("2024-01-23 18:47:25", utc=True)
    end   = pd.to_datetime("2024-01-23 19:17:25", utc=True)

# 1) Filter log.csv to candidate services
df_log_cand = df_log[df_log['cmdb_id'].isin(candidates)].copy()

# If empty, prepare empty outputs
if df_log_cand.empty:
    log_agg_top20 = pd.DataFrame(columns=['cmdb_id','log_name','global_p95','total_points',
                                          'points_in_window','max_value_in_window','mean_value_in_window','earliest_anom_ts_utc'])
else:
    # Compute global_p95 and total_points on full-series per (cmdb_id, log_name)
    grp_full = df_log_cand.groupby(['cmdb_id','log_name'])['value']
    agg_full = grp_full.agg(global_p95=lambda x: x.quantile(0.95), total_points='count').reset_index()
    agg_full['global_p95'] = agg_full['global_p95'].astype(float)

    # Windowed data and aggregates
    df_log_win = df_log_cand[(df_log_cand['ts'] >= start) & (df_log_cand['ts'] <= end)].copy()
    win_agg = df_log_win.groupby(['cmdb_id','log_name'])['value'].agg(
        points_in_window='count',
        max_value_in_window='max',
        mean_value_in_window='mean'
    ).reset_index()

    # Earliest anomaly in window where value > global_p95
    if not df_log_win.empty:
        df_log_win = df_log_win.merge(agg_full[['cmdb_id','log_name','global_p95']], on=['cmdb_id','log_name'], how='left')
        anom = df_log_win[df_log_win['value'] > df_log_win['global_p95']].copy()
        if not anom.empty:
            anom_earliest = anom.groupby(['cmdb_id','log_name'])['ts'].min().reset_index().rename(columns={'ts':'earliest_anom_ts_utc'})
        else:
            anom_earliest = pd.DataFrame(columns=['cmdb_id','log_name','earliest_anom_ts_utc'])
    else:
        anom_earliest = pd.DataFrame(columns=['cmdb_id','log_name','earliest_anom_ts_utc'])

    # Merge full + window + anomaly
    log_agg = agg_full.merge(win_agg, on=['cmdb_id','log_name'], how='left').merge(anom_earliest, on=['cmdb_id','log_name'], how='left')

    # Fill missing window counts with 0 and keep mean/max as NaN if absent
    log_agg['points_in_window'] = log_agg['points_in_window'].fillna(0).astype(int)
    # Format earliest_anom_ts_utc as ISO string or None
    log_agg['earliest_anom_ts_utc'] = log_agg['earliest_anom_ts_utc'].apply(lambda x: None if pd.isna(x) else x.isoformat())

    # Select and order columns
    log_agg = log_agg[['cmdb_id','log_name','global_p95','total_points',
                       'points_in_window','max_value_in_window','mean_value_in_window','earliest_anom_ts_utc']]

    # 3) Sort and limit top 20 by points_in_window desc then max_value_in_window desc
    log_agg_top20 = log_agg.sort_values(by=['points_in_window','max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

# 4) Search error_logs.csv for candidate cmdb_ids within the incident window, return up to 20 rows with ts UTC, cmdb_id, message
err_hits = df_err[df_err['cmdb_id'].isin(candidates) & (df_err['ts'] >= start) & (df_err['ts'] <= end)].copy()
if not err_hits.empty:
    err_hits = err_hits.sort_values('ts').loc[:, ['ts','cmdb_id','message']].head(20)
    # format ts as ISO strings for compact display
    err_hits['ts'] = err_hits['ts'].dt.strftime('%Y-%m-%dT%H:%M:%S%z')
else:
    err_hits = err_hits.loc[:, ['ts','cmdb_id','message']].head(0)

# Final compact outputs
log_agg_top20, err_hits

```
Out[5]:
```
Summary of log analysis for candidate services (incident window 2024-01-23 18:47:25 → 2024-01-23 19:17:25 UTC):

Key aggregated findings (top rows):

- ts-travel-service — log.row_count
  - global_p95 = 2071.6, total_points = 25
  - points_in_window = 25, max_in_window = 2124, mean_in_window = 1557.24
  - earliest value > global_p95: 2024-01-23T18:55:00Z

- ts-route-service — log.row_count
  - global_p95 = 1085.4, total_points = 25
  - points_in_window = 25, max_in_window = 1151, mean_in_window = 936.00
  - earliest anomaly: 2024-01-23T18:55:00Z

- ts-order-service — log.row_count
  - global_p95 = 468.4, total_points = 25
  - points_in_window = 25, max_in_window = 473, mean_in_window = 349.72
  - earliest anomaly: 2024-01-23T18:55:00Z

- ts-auth-service — log.row_count
  - global_p95 = 337.6, total_points = 25
  - points_in_window = 25, max_in_window = 344, mean_in_window = 299.28
  - earliest anomaly: 2024-01-23T18:52:00Z

- log.error_count (for ts-auth, ts-order, ts-route, ts-travel)
  - global_p95 = 0.0, total_points = 25
  - points_in_window = 25, max_in_window = 0, mean_in_window = 0.0
  - earliest_anom: None (no values exceeded p95)

Error logs:
- No rows in error_logs.csv for any candidate cmdb_id within the incident window.

Interpretation:
- All candidate services show sustained high log.row_count during the incident window; ts-travel and ts-route have the largest absolute row_count spikes (exceeding their p95 around 18:55 UTC). 
- ts-auth-service shows an earlier row_count anomaly at ~18:52 UTC.
- log.error_count stays at zero (no obvious log-error signal).
- No error_log entries were recorded for these services in the window.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  global_p95  total_points  points_in_window  max_value_in_window  mean_value_in_window       earliest_anom_ts_utc
0  ts-travel-service    log.row_count      2071.6            25                25                 2124               1557.24  2024-01-23T18:55:00+00:00
1   ts-route-service    log.row_count      1085.4            25                25                 1151                936.00  2024-01-23T18:55:00+00:00
2   ts-order-service    log.row_count       468.4            25                25                  473                349.72  2024-01-23T18:55:00+00:00
3    ts-auth-service    log.row_count       337.6            25                25                  344                299.28  2024-01-23T18:52:00+00:00
4    ts-auth-service  log.error_count         0.0            25                25                    0                  0.00                       None
5   ts-order-service  log.error_count         0.0            25                25                    0                  0.00                       None
6   ts-route-service  log.error_count         0.0            25                25                    0                  0.00                       None
7  ts-travel-service  log.error_count         0.0            25                25                    0                  0.00                       None, Empty DataFrame
Columns: [ts, cmdb_id, message]
Index: [])```
```